In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib

START_YEAR = 2019
embedder = SentenceTransformer('all-MiniLM-L6-v2')

CIHR

In [ ]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [ ]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)


In [ ]:
CIHR_DATA["Main_Discipline"].value_counts()

In [ ]:
tmp_data = CIHR_DATA.copy()

tmp_data["Area_of_Research"].value_counts()

NSERC

In [2]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [3]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for Main Discipline

In [ ]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not available"]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Advancement of knowledge                   500
Energy resources                           500
The socioeconomic objective available      500
Transportation systems and services        500
Manufacturing processes and products       500
Agriculture and primary food production    500
Northern development                       500
Health, education and social services      500
Natural resources (economic aspects)       500
Environment                                500
Information and communication services     500
Construction, urban and rural planning     500
Commercial services                        400
Name: count, dtype: int64

In [ ]:
# tmp_data['text_input_md'] = tmp_data['Title'].fillna('') + ':' + tmp_data['Area_of_Research'].fillna('') + ':' + tmp_data['Keywords'].fillna('')

In [7]:
X_data = tmp_data['Title']
y_data = tmp_data['Main_Discipline']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, stratify=y_data)

X_train_embed = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embed = embedder.encode(X_test.tolist(), show_progress_bar=True)

clf = GradientBoostingClassifier(n_estimators=300)

#Train Model
clf.fit(X_train_embed, y_train)

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

GradientBoostingClassifier(n_estimators=300)

In [8]:
print(classification_report(y_test, clf.predict(X_test_embed)))

                                         precision    recall  f1-score   support

               Advancement of knowledge       0.34      0.27      0.30       125
Agriculture and primary food production       0.65      0.70      0.67       125
                    Commercial services       0.58      0.60      0.59       100
 Construction, urban and rural planning       0.64      0.62      0.63       125
                       Energy resources       0.60      0.64      0.62       125
                            Environment       0.48      0.42      0.45       125
  Health, education and social services       0.53      0.58      0.56       125
 Information and communication services       0.55      0.66      0.60       125
   Manufacturing processes and products       0.30      0.31      0.30       125
   Natural resources (economic aspects)       0.63      0.60      0.61       125
                   Northern development       0.40      0.38      0.39       125
  The socioeconomic objecti

In [9]:
joblib.dump(clf, 'models/NSERC_MD_model.pkl')

['models/NSERC_MD_model.pkl']

Model for Area of Research

In [10]:
translator = GoogleTranslator(target="en")

tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle


val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 50].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

for class_name in classes:
    tmp_data["Area_of_Research"] = tmp_data["Area_of_Research"].replace(class_name, translator.translate(class_name))

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].str.replace(r' \(.*?\)', '', regex=True)

tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data[tmp_data["Area_of_Research"] != "Not available"]

tmp_data["Area_of_Research"].value_counts()


Area_of_Research
Analytical chemistry              500
Psychology                        500
Molecular biology                 500
Evolution and ecology             500
Astronomy and astrophysics        500
                                 ... 
Animal nutrition and husbandry    105
Photonics                         104
Enzymes                           102
Fuel and energy technology        102
Database management               101
Name: count, Length: 135, dtype: int64

In [ ]:
# tmp_data['text_input_ar'] = tmp_data['Title'].fillna('') + ' ' + tmp_data['Main_Discipline'].fillna('') + ' ' + tmp_data['Keywords'].fillna('')

In [ ]:
X_data = tmp_data['Title']
y_data = tmp_data['Area_of_Research']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, stratify=y_data)

X_train_embed = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embed = embedder.encode(X_test.tolist(), show_progress_bar=True)

clf = GradientBoostingClassifier()

#Train Model
clf.fit(X_train_embed, y_train)

Batches:   0%|          | 0/893 [00:00<?, ?it/s]

Batches:   0%|          | 0/298 [00:00<?, ?it/s]

In [ ]:
print(classification_report(y_test, clf.predict(X_test_embed)))

In [ ]:
joblib.dump(clf, 'models/NSERC_AR_model.pkl')

SSHRC

In [ ]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [ ]:
grant_descriptors = [
    "Title-Titre", "Main_Discipline", "Area_of_Research", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

In [ ]:
SSHRC_DATA["Main_Discipline"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]


tmp_data["Main_Discipline"].value_counts()


In [ ]:
X_data = tmp_data['Title']
y_data = tmp_data['Main_Discipline']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, stratify=y_data)

X_train_embed = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embed = embedder.encode(X_test.tolist(), show_progress_bar=True)

clf = GradientBoostingClassifier()

#Train Model
clf.fit(X_train_embed, y_train)

In [ ]:
print(classification_report(y_test, clf.predict(X_test_embed)))

In [ ]:
joblib.dump(clf, "models/SSHRC_MD_model.pkl")

Model for Area of Research

In [ ]:
SSHRC_DATA["Area_of_Research"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data[~(tmp_data['Area_of_Research'].isin(["Not Subject to Research Classification", "Not Specified"]))]

tmp_data["Area_of_Research"].value_counts()


In [ ]:
X_data = tmp_data['Title']
y_data = tmp_data['Area_of_Research']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, stratify=y_data)

X_train_embed = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embed = embedder.encode(X_test.tolist(), show_progress_bar=True)

clf = GradientBoostingClassifier()

#Train Model
clf.fit(X_train_embed, y_train)

In [ ]:
print(classification_report(y_test, clf.predict(X_test_embed)))

In [ ]:
joblib.dump(clf, 'models/SSHRC_AR_model.pkl')

In [ ]:
embedder.save("models/embedder")